In [ ]:
print("HALOO")

HALOO


In [ ]:
from google.colab import drive
import os
import pandas as pd

drive.mount("/content/drive")

path_excel = "/content/drive/MyDrive/online_retail_II.xlsx"
BASE = "/content/drive/MyDrive/retail-project"

print("File Excel ada?", os.path.exists(path_excel))

os.makedirs(f"{BASE}/data", exist_ok=True)
os.makedirs(f"{BASE}/reports", exist_ok=True)

sheets = pd.read_excel(path_excel, sheet_name=None)
df = pd.concat(sheets.values(), ignore_index=True)
df.to_csv(f"{BASE}/data/retail_raw_as_csv.csv", index=False)
print(df.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
File Excel ada? True
(1067371, 8)


In [ ]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [ ]:
df.tail()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France
1067370,581587,POST,POSTAGE,1,2011-12-09 12:50:00,18.00,12680.0,France


In [ ]:
df = df.rename(columns={
    "Invoice": "invoice", "StockCode": "stock_code", "Description": "description",
    "Quantity": "quantity", "InvoiceDate": "invoice_date", "Price": "unit_price",
    "Customer ID": "customer_id", "Country": "country",
})
df["stock_code"] = df["stock_code"].astype(str).str.strip()

log = []
def log_step(step, reason, before, after):
    log.append({"step": step, "reason": reason, "rows_before": before, "rows_after": after})

log_step("Load raw data (both sheets combined)", "Two sheets cover Dec 2009 - Dec 2011", len(df), len(df))
print(df.shape)

(1067371, 8)


Cell 2: duplikat

In [ ]:
n = len(df)
df = df.drop_duplicates()
log_step("Remove exact duplicates",
         "Sheets overlap on 1-9 Dec 2010; identical rows would be counted twice", n, len(df))
print(n - len(df), "rows removed")

34335 rows removed


Cell 3: kode non-produk

In [ ]:
non_product = ["POST", "D", "M", "DOT", "BANK CHARGES", "AMAZONFEE", "ADJUST", "ADJUST2",
               "S", "B", "CRUK", "C2", "TEST001", "TEST002", "PADS"]
code = df["stock_code"].str.upper()
is_non_product = code.isin(non_product) | code.str.startswith("GIFT_")

n = len(df)
df = df[~is_non_product]
log_step("Remove non-product codes",
         "Postage, fees, discounts, manual entries and gift vouchers are not product sales", n, len(df))
print(n - len(df), "rows removed")

5818 rows removed


Cell 4: harga nol atau negatif

In [ ]:
n = len(df)
df = df[df["unit_price"] > 0]
log_step("Remove price <= 0",
         "Free items and accounting adjustments, not real sales", n, len(df))
print(n - len(df), "rows removed")

5964 rows removed


Cell 5: pisahkan pembatalan (returns)

In [ ]:
is_cancel = df["invoice"].astype(str).str.startswith("C")
returns = df[is_cancel].copy()
n = len(df)
df = df[~is_cancel].copy()
log_step("Split cancelled orders into returns table",
         "Invoice starts with C. Kept separately so we can calculate net revenue", n, len(df))

n = len(df)
df = df[df["quantity"] > 0].copy()
log_step("Remove quantity <= 0 in sales",
         "Stock write-offs and corrections without a customer", n, len(df))
print("sales:", len(df), "| returns:", len(returns))

sales: 1003340 | returns: 17914


In [ ]:
df["description"] = df["description"].str.strip()
returns["description"] = returns["description"].str.strip()

main_desc = (df.dropna(subset=["description"])
               .groupby("stock_code")["description"]
               .agg(lambda s: s.mode().iloc[0]))
df["description"] = df["stock_code"].map(main_desc)
returns["description"] = returns["stock_code"].map(main_desc)

for d in (df, returns):
    d["customer_id"] = d["customer_id"].astype("Int64")
    d["revenue"] = d["quantity"] * d["unit_price"]

log_step("Standardize descriptions",
         "One name per product code (most frequent name)", len(df), len(df))
log_step("Keep rows with missing Customer ID",
         f"{df['customer_id'].isna().sum()} rows. Valid for sales by product/country/time; excluded later from customer analysis (RFM, repeat rate)",
         len(df), len(df))

In [ ]:
cols = ["invoice", "description", "quantity", "unit_price", "revenue"]
print(df.nlargest(5, "revenue")[cols])
print(returns.nsmallest(5, "revenue")[cols])

        invoice                          description  quantity  unit_price  \
1065882  581483          PAPER CRAFT , LITTLE BIRDIE     80995        2.08   
587080   541431       MEDIUM CERAMIC TOP STORAGE JAR     74215        1.04   
748132   556444           PICNIC BASKET WICKER SMALL        60      649.50   
432176   530715  ROTATING SILVER ANGELS T-LIGHT HLDR      9360        1.69   
228042   511465                   PINK PAPER PARASOL      3500        2.55   

          revenue  
1065882  168469.6  
587080    77183.6  
748132    38970.0  
432176    15818.4  
228042     8925.0  
         invoice                         description  quantity  unit_price  \
1065883  C581484         PAPER CRAFT , LITTLE BIRDIE    -80995        2.08   
587085   C541433      MEDIUM CERAMIC TOP STORAGE JAR    -74215        1.04   
685606   C550456  FAIRY CAKE FLANNEL ASSORTED COLOUR     -3114        2.10   
685604   C550456  WHITE HANGING HEART T-LIGHT HOLDER     -1930        2.55   
685602   C550456     

In [ ]:
log_step("Outliers: reviewed, not removed",
         "Large orders may be genuine wholesale; cancelled ones are netted out via returns", len(df), len(df))

In [ ]:
assert df["stock_code"].notna().all()
print("null description:", df["description"].isna().sum())   # harus jauh dari 882 ribu

null description: 0


In [ ]:
print(df.shape, returns.shape)
print(df.isna().sum())
print(df["invoice_date"].min(), df["invoice_date"].max())

gross = df["revenue"].sum()
ret = returns["revenue"].sum()
print(f"Gross sales: {gross:,.0f} | Returns: {ret:,.0f} | Net revenue: {gross + ret:,.0f}")

df.to_csv(f"{BASE}/data/sales_clean.csv", index=False)
returns.to_csv(f"{BASE}/data/returns_clean.csv", index=False)
pd.DataFrame(log).to_csv(f"{BASE}/reports/cleaning_log.csv", index=False)
pd.DataFrame(log)

(1003340, 9) (17914, 9)
invoice              0
stock_code           0
description          0
quantity             0
invoice_date         0
unit_price           0
customer_id     226761
country              0
revenue              0
dtype: int64
2009-12-01 07:45:00 2011-12-09 12:50:00
Gross sales: 19,643,862 | Returns: -716,426 | Net revenue: 18,927,436


,step,reason,rows_before,rows_after
0,Load raw data (both sheets combined),Two sheets cover Dec 2009 - Dec 2011,1067371,1067371
1,Remove exact duplicates,Sheets overlap on 1-9 Dec 2010; identical rows...,1067371,1033036
2,Remove non-product codes,"Postage, fees, discounts, manual entries and g...",1033036,1027218
3,Remove price <= 0,"Free items and accounting adjustments, not rea...",1027218,1021254
4,Split cancelled orders into returns table,Invoice starts with C. Kept separately so we c...,1021254,1003340
5,Remove quantity <= 0 in sales,Stock write-offs and corrections without a cus...,1003340,1003340
6,Standardize descriptions,One name per product code (most frequent name),1003340,1003340
7,Keep rows with missing Customer ID,226761 rows. Valid for sales by product/countr...,1003340,1003340
8,"Outliers: reviewed, not removed",Large orders may be genuine wholesale; cancell...,1003340,1003340
